# `search_test.ipynb` — Phase 3 Qdrant 검색 데모

> **목적**: Phase 3 Stage 2b 로 적재된 `audit_standards_회계감사기준_2025` collection (8,626 points) 에 대해
> 2-단계 검색 파이프라인을 시연하고, 동시에 `docs/checkpoint_3_prep.md §9.5` 의 10 seed query
> 헤더 bias 실측을 수행합니다.

## 사용 전 체크리스트

1. `docker compose up -d` — Qdrant 가 `http://localhost:6333` 에 떠 있어야 합니다.
2. `.env` 파일에 `UPSTAGE_API_KEY` 설정이 되어 있어야 합니다 (query 임베딩 10건 신규 호출).
3. 프로젝트 루트에서 `source .venv/bin/activate && uv sync` 로 설치 완료.
4. Collection 은 이미 적재 완료 상태여야 합니다. 확인:
   ```bash
   curl -sS http://localhost:6333/collections/audit_standards_회계감사기준_2025 | jq '.result.points_count'
   # → 8626 expected
   ```

## 노트북 구조

| §  | 내용 |
|---:|---|
| 1  | 환경 설정 — Qdrant client + Embedder 초기화 |
| 2  | Sanity check — collection 상태, named vectors, payload index 확인 |
| 3  | 단일 query 검색 (passage vector) — 한 건 top-5 확인 |
| 4  | 2-단계 검색 데모 (summary → standard_id filter → passage) |
| 5  | 10 seed query 일괄 실행 — §9.5 header bias 실측 |
| 6  | Header bias 판정 — 조건 A (co-occurrence) + 조건 B (cosine Δ) |
| 7  | 결과 저장 — `output/search_demo_results.json` |

각 셀은 독립 실행 가능하도록 작성되어 있어, 특정 검색만 확인하고 싶다면 §3 만 실행해도 됩니다.

## 1. 환경 설정

Qdrant client 와 Embedder 를 준비합니다. Embedder 는 `.embed_cache.sqlite` 를 공유 (passage 8,590건 + summary 36건 warm-up 완료).

In [1]:
import json
import os
import sys
import time
from itertools import combinations
from pathlib import Path

# 프로젝트 루트를 sys.path 에 추가 (notebook 에서 editable install 없이도 동작)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from audit_parser.ingest.embedder import Embedder
from audit_parser.ingest.qdrant_writer import (
    KIND_STANDARD_SUMMARY,
    VECTOR_PASSAGE,
    VECTOR_SUMMARY,
    chunk_id_to_point_id,
)
from qdrant_client import QdrantClient

QDRANT_URL = os.environ.get("QDRANT_URL", "http://localhost:6333")
CACHE_PATH = PROJECT_ROOT / ".embed_cache.sqlite"

client = QdrantClient(url=QDRANT_URL, timeout=30)
embedder = Embedder(cache_path=CACHE_PATH)

print(f"Qdrant:    {QDRANT_URL}")
print(f"Cache:     {CACHE_PATH} (exists={CACHE_PATH.exists()})")
print(f"API key:   {'set' if os.environ.get('UPSTAGE_API_KEY') else 'MISSING'}")
print("(COLLECTION 은 다음 §1.5 CONFIG 셀에서 설정됩니다.)")

Qdrant:    http://localhost:6333
Cache:     /home/shin/Project/_AuditStandard_parsing/.embed_cache.sqlite (exists=True)
API key:   set
(COLLECTION 은 다음 §1.5 CONFIG 셀에서 설정됩니다.)


## 1.5 CONFIG — 첫 셀에서 변경 가능한 파라미터

아래 상수만 변경해도 노트북 전체 동작이 달라집니다. 기본값은 Phase 3 권장값.

| 파라미터 | 기본 | 설명 |
|---|---|---|
| `TOP_K` | 5 | 단일 query 검색 시 반환 chunk 수 |
| `N_STANDARDS` | 3 | 2단계 검색 Stage1 후보 standard 수 |
| `K_PER_STANDARD` | 5 | 2단계 검색 Stage2 per-standard top-k |
| `COLLECTION` | `audit_standards_회계감사기준_2025` | CLAUDE.md §5 기본 collection |
| `EXAMPLE_QUERIES` | 5건 | 기본 자연어 쿼리 예시 (자유 추가) |
| `DEFAULT_FILTER` | `None` | payload filter (예: `standard_id="ISA-315"` 한정) |

In [2]:
from __future__ import annotations
from qdrant_client.models import Filter, FieldCondition, MatchValue

# --- 사용자 편집 영역 시작 ---
TOP_K = 5                        # 단일 query 반환 수
N_STANDARDS = 3                  # 2단계 Stage1 후보 standard 수
K_PER_STANDARD = 5               # 2단계 Stage2 per-standard top-k
# Collection name — CLAUDE.md §5. Phase 4 에서 다른 기준서 (ISQM 1 등) 시 변경.
COLLECTION = "audit_standards_회계감사기준_2025"

EXAMPLE_QUERIES: list[str] = [
    "감사증거의 충분성과 적합성",         # 정의/개념
    "중요성 판단 기준",                  # ISA-320 핵심
    "위험평가절차와 중요한 왜곡표시 위험",   # ISA-315 핵심
    "감사의견 변형 — 한정의견의 상황",      # ISA-705 핵심
    "계속기업으로서의 존속능력 평가",        # ISA-570 핵심
]

# payload filter — None 이면 전체 검색. 예시 주석 풀어서 사용.
DEFAULT_FILTER: Filter | None = None
# DEFAULT_FILTER = Filter(must=[FieldCondition(key="standard_id", match=MatchValue(value="ISA-315"))])
# DEFAULT_FILTER = Filter(must=[FieldCondition(key="section", match=MatchValue(value="appendix"))])
# DEFAULT_FILTER = Filter(must=[FieldCondition(key="is_application_guidance", match=MatchValue(value=True))])
# --- 사용자 편집 영역 끝 ---

print(f"TOP_K            = {TOP_K}")
print(f"N_STANDARDS      = {N_STANDARDS}")
print(f"K_PER_STANDARD   = {K_PER_STANDARD}")
print(f"COLLECTION       = {COLLECTION}")
print(f"EXAMPLE_QUERIES  = {len(EXAMPLE_QUERIES)}건")
print(f"DEFAULT_FILTER   = {DEFAULT_FILTER}")

TOP_K            = 5
N_STANDARDS      = 3
K_PER_STANDARD   = 5
COLLECTION       = audit_standards_회계감사기준_2025
EXAMPLE_QUERIES  = 5건
DEFAULT_FILTER   = None


## 2. Sanity check — collection 상태

8,590 chunk + 36 summary = **8,626 points** 기대. named vectors 는 `passage` / `summary`, 둘 다 4096d cosine.

In [3]:
info = client.get_collection(COLLECTION)
print(f"points_count:          {info.points_count}")
print(f"indexed_vectors_count: {info.indexed_vectors_count}")
print(f"status:                {info.status}")
print(f"vector names:          {list(info.config.params.vectors.keys())}")
for vname, vparams in info.config.params.vectors.items():
    print(f"  {vname}: size={vparams.size} distance={vparams.distance}")

assert info.points_count == 8626, f"expected 8626 points, got {info.points_count}"
print("\nOK — collection 적재 완료 상태.")

points_count:          8626
indexed_vectors_count: 17252
status:                green
vector names:          ['passage', 'summary']
  passage: size=4096 distance=Cosine
  summary: size=4096 distance=Cosine

OK — collection 적재 완료 상태.


In [7]:
# summary point 36건 확인 (kind=standard_summary payload filter)
summary_filter = Filter(
    must=[FieldCondition(key="kind", match=MatchValue(value=KIND_STANDARD_SUMMARY))]
)
summary_count = client.count(COLLECTION, count_filter=summary_filter, exact=True)
print(f"standard_summary points: {summary_count.count} (expected 36)")

# 샘플 1건 payload 확인
scroll_res, _ = client.scroll(
    collection_name=COLLECTION,
    scroll_filter=summary_filter,
    limit=1,
    with_payload=True,
)
sample = scroll_res[0].payload
print(f"\nsample summary payload keys: {sorted(sample.keys())}")
print(f"standard_id: {sample['standard_id']}")
print(f"content_text[:120]: {sample['content_text'][:120]}...")

standard_summary points: 36 (expected 36)

sample summary payload keys: ['appendix_index', 'authority', 'authority_base', 'chunk_id', 'chunk_index', 'chunk_of', 'content_markdown', 'content_text', 'content_text_hash', 'embedded_at', 'embedding_model', 'heading_trail', 'heading_trail_hash', 'is_application_guidance', 'kind', 'paragraph_id', 'parent_paragraph_id', 'part_of', 'section', 'source_file', 'source_idx', 'standard_id', 'standard_no', 'table_cells', 'token_estimate']
standard_id: ISA-300
content_text[:120]: 1. 이 감사기준서는 재무제표감사를 계획할 감사인의 책임을 다룬다. 이 감사기준서는 계속감사의 관점에서 작성되었으며, 초도감사의 추가적인 고려사항은 별도로 식별되고 있다....


## 3. 단일 query 검색 — passage vector

가장 기본적인 의미 검색: query 를 `embedding-query` 모델로 임베딩 → passage named vector 로 top-5 검색.

> **캐시 노트**: 동일 query 를 재실행하면 Upstage API 호출 없이 SQLite 캐시에서 읽어옵니다 (`embedder.stats.cached_hits` 증가).

In [5]:
def search_top_k(
    query_text: str,
    k: int = TOP_K,
    filter_: Filter | None = None,
    vector_name: str = VECTOR_PASSAGE,
) -> list[dict]:
    """query → embed_query → Qdrant top-k. 결과는 dict 리스트.

    qdrant-client >=1.10: ``query_points(query=[...], using=<name>, ...)`` 사용.
    """
    qres = embedder.embed_query(query_text)
    effective_filter = filter_ if filter_ is not None else DEFAULT_FILTER
    response = client.query_points(
        collection_name=COLLECTION,
        query=list(qres.vector),
        using=vector_name,
        query_filter=effective_filter,
        limit=k,
        with_payload=True,
    )
    return [
        {
            "rank": i + 1,
            "score": round(h.score, 4),
            "chunk_id": h.payload["chunk_id"],
            "standard_id": h.payload["standard_id"],
            "kind": h.payload["kind"],
            "section": h.payload.get("section"),
            "heading_trail": h.payload.get("heading_trail", []),
            "content_preview": h.payload["content_text"][:120],
        }
        for i, h in enumerate(response.points)
    ]

# 기본 예시 쿼리 1건 실행 (아래 outputs 에 미리 렌더됨)
demo_query = EXAMPLE_QUERIES[0]
print(f"Query: {demo_query}\n")
for row in search_top_k(demo_query, k=TOP_K):
    print(f"#{row['rank']}  score={row['score']}  [{row['standard_id']}] {row['chunk_id']}")
    print(f"      {row['content_preview']}...\n")

Query: 감사증거의 충분성과 적합성

#1  score=0.5221  [ISA-700] ISA-700:requirements:0f204eb7:(d)
      입수한 감사증거가 감사의견의 근거를 제공하기에 충분하고 적합하다고 감사인이 믿는지 여부에 대한 기술...

#2  score=0.5221  [ISA-1200] ISA-1200:conclusion_reporting:e7549647:(iv)#11010
      입수한 감사증거가 감사의견의 근거를 제공하기에 충분하고 적합하다고 감사인이 믿는지 여부에 대한 기술...

#3  score=0.5221  [ISA-1100] ISA-1100:requirements:b5fda7bb:(iv)
      입수한 감사증거가 감사의견의 근거를 제공하기에 충분하고 적합하다고 감사인이 믿는지 여부에 대한 기술...

#4  score=0.5199  [ISA-540] ISA-540:application:df858097:paragraph_body#5655
      이러한 상황에서 감사증거의 충분성과 적합성은 정보의 정확성과 완전성에 대한 통제의 효과성에 의존할 수 있다....

#5  score=0.5184  [ISA-1200] ISA-1200:conclusion_reporting:b3a93760:(b)#11048
      입수한 감사증거가 감사의견의 근거를 제공하는 데 충분하고 적합한지 여부에 관한 기술...



## 4. 2-단계 검색 — summary → standard_id filter → passage

**설계 근거** (`qdrant_writer.py F1 — Critic 채택`):
- Chunk point 8,590건에 summary vector 를 복제하면 ~140MB 낭비.
- 대신 standard 당 1 건 `kind='standard_summary'` point (36건) 를 별도 저장.
- 검색 흐름:
  1. Query → `summary` named vector 로 summary 36건 중 top-N (e.g. 3) standard 선정.
  2. 선정된 standard_id 로 payload filter 를 걸어 `passage` named vector top-k 재검색.

이 방식은 "관련 기준서 후보" 를 먼저 좁힌 뒤 세부 chunk 를 찾으므로, 전체 8,626 point 를 passage vector 로 훑는 것보다 **관련 standard 유지력이 좋습니다** (cross-standard noise 감소).

In [6]:
def two_stage_search(
    query_text: str,
    n_standards: int = N_STANDARDS,
    k_per_standard: int = K_PER_STANDARD,
) -> dict:
    """Stage1: summary top-N 으로 후보 standard 선정.
       Stage2: 후보 standard_id 내에서 passage top-k 검색."""
    qres = embedder.embed_query(query_text)
    qvec = list(qres.vector)

    # Stage 1 — summary 36건 중 top-N standard 선정
    stage1 = client.query_points(
        collection_name=COLLECTION,
        query=qvec,
        using=VECTOR_SUMMARY,
        query_filter=Filter(
            must=[FieldCondition(key="kind", match=MatchValue(value=KIND_STANDARD_SUMMARY))]
        ),
        limit=n_standards,
        with_payload=True,
    )
    candidate_standards = [
        (h.payload["standard_id"], round(h.score, 4)) for h in stage1.points
    ]

    # Stage 2 — 각 standard 에서 passage top-k
    per_standard_hits: dict[str, list[dict]] = {}
    for sid, _score in candidate_standards:
        std_filter = Filter(
            must=[FieldCondition(key="standard_id", match=MatchValue(value=sid))],
            must_not=[
                FieldCondition(key="kind", match=MatchValue(value=KIND_STANDARD_SUMMARY))
            ],
        )
        stage2 = client.query_points(
            collection_name=COLLECTION,
            query=qvec,
            using=VECTOR_PASSAGE,
            query_filter=std_filter,
            limit=k_per_standard,
            with_payload=True,
        )
        per_standard_hits[sid] = [
            {
                "rank": i + 1,
                "score": round(h.score, 4),
                "chunk_id": h.payload["chunk_id"],
                "section": h.payload.get("section"),
                "preview": h.payload["content_text"][:100],
            }
            for i, h in enumerate(stage2.points)
        ]

    return {
        "query": query_text,
        "stage1_candidates": candidate_standards,
        "stage2_hits": per_standard_hits,
    }

# 2-단계 검색 데모 — CONFIG.N_STANDARDS / CONFIG.K_PER_STANDARD 사용
two_stage_demo = EXAMPLE_QUERIES[2]   # "위험평가절차와 중요한 왜곡표시 위험"
result = two_stage_search(two_stage_demo)
print(f"Query: {result['query']}\n")
print(f"Stage 1 — 후보 standard (summary top-{N_STANDARDS}):")
for sid, score in result["stage1_candidates"]:
    print(f"  {sid:10s}  score={score}")
print(f"\nStage 2 — 각 standard 내 passage top-{K_PER_STANDARD}:")
for sid, hits in result["stage2_hits"].items():
    print(f"\n  [{sid}]")
    for h in hits:
        print(f"    #{h['rank']} score={h['score']}  {h['chunk_id']}")
        print(f"         {h['preview']}...")


Query: 위험평가절차와 중요한 왜곡표시 위험

Stage 1 — 후보 standard (summary top-3):
  ISA-330     score=0.439
  ISA-315     score=0.4365
  ISA-450     score=0.4068

Stage 2 — 각 standard 내 passage top-5:

  [ISA-330]
    #1 score=0.5612  ISA-330:application:7ec66b6b:bullet#3919
         평가된 중요왜곡표시위험...
    #2 score=0.4819  ISA-330:application:364c4f8d:bullet#3841
         경영진주장 수준의 평가된 중요왜곡표시위험의 유의성...
    #3 score=0.4578  ISA-330:application:040930bb:A1.
         재무제표 수준의 평가된 중요왜곡표시위험에 대처하기 위한 전반적인 대응에는 다음 사항이 포함될 것이다....
    #4 score=0.4395  ISA-330:application:3ef062a2:bullet#3937
         감사의 전반적인 검토단계에서 수행된 분석적절차는 이전에 인식하지 못한 중요왜곡표시위험을 나타낼 수도 있음...
    #5 score=0.4357  ISA-330:application:0ef32c4e:A4.
         경영진주장 수준에서 식별된 중요왜곡표시위험에 대한 감사인의 평가는 감사인이 추가감사절차를 설계하고 수행하기 위한 적합한 감사접근방법을 고려하는 데 하나의 근거를 제공한다. 감사인은...

  [ISA-315]
    #1 score=0.5102  ISA-315:definitions:56d3af17:(j)
         위험평가절차 - 재무제표 및 경영진주장 수준에서 부정이나 오류로 인한 중요왜곡표시위험을 식별하고 평가하기 위하여 수행하는 감사절차...
    #2 score=0.5021  ISA-315:requirem

## 5. 10 seed query 일괄 실행 — §9.5 header bias 실측

`docs/checkpoint_3_prep.md §1.3` 에서 확정한 10 seed query 로 ISA-1200 용어정의 3-part split 의 header 복제 bias 를 측정합니다.

### Target chunk (3-part split)
| chunk_id | part |
|---|---|
| `ISA-1200:appendix:d3ec59bd:table#11079`   | part 0 |
| `ISA-1200:appendix:d3ec59bd:table#11079#1` | part 1 |
| `ISA-1200:appendix:d3ec59bd:table#11079#2` | part 2 |

In [7]:
SEED_QUERIES: list[tuple[str, str]] = [
    ("A", "용어의 정의"),
    ("A", "감사기준서 용어 정의 전체 목록"),
    ("A", "용어 정의 사전"),
    ("B", "감사증거의 적합성과 충분성"),
    ("B", "경영진주장이란 무엇인가"),
    ("C", "숙련된 감사인의 요건"),
    ("C", "실증절차의 정의"),
    ("D", "특수관계자의 정의"),
    ("D", "표본감사와 표본단위"),
    ("E", "감사위원회와 지배기구의 커뮤니케이션"),
]

TARGET_CHUNKS = {
    "ISA-1200:appendix:d3ec59bd:table#11079",
    "ISA-1200:appendix:d3ec59bd:table#11079#1",
    "ISA-1200:appendix:d3ec59bd:table#11079#2",
}

seed_results: list[dict] = []
t0 = time.perf_counter()
for category, query in SEED_QUERIES:
    top5 = search_top_k(query, k=5)
    hit_ids = [row["chunk_id"] for row in top5]
    target_hits = [cid for cid in hit_ids if cid in TARGET_CHUNKS]
    seed_results.append(
        {
            "category": category,
            "query": query,
            "top5": top5,
            "target_hits": target_hits,
            "co_occur_2plus": len(target_hits) >= 2,
        }
    )
elapsed = time.perf_counter() - t0

print(f"10 query 실행 완료 ({elapsed:.2f}s)")
print(f"embedder api_calls: {embedder.stats.api_calls}, cached_hits: {embedder.stats.cached_hits}")
print("\n=== 결과 요약 ===")
for r in seed_results:
    flag = "⚠️ CO-OCCUR" if r["co_occur_2plus"] else ""
    print(f"[{r['category']}] {r['query']:30s}  targets_in_top5={len(r['target_hits'])}  {flag}")

10 query 실행 완료 (0.37s)
embedder api_calls: 0, cached_hits: 12

=== 결과 요약 ===
[A] 용어의 정의                          targets_in_top5=0  
[A] 감사기준서 용어 정의 전체 목록               targets_in_top5=0  
[A] 용어 정의 사전                        targets_in_top5=1  
[B] 감사증거의 적합성과 충분성                  targets_in_top5=0  
[B] 경영진주장이란 무엇인가                    targets_in_top5=0  
[C] 숙련된 감사인의 요건                     targets_in_top5=0  
[C] 실증절차의 정의                        targets_in_top5=0  
[D] 특수관계자의 정의                       targets_in_top5=0  
[D] 표본감사와 표본단위                      targets_in_top5=0  
[E] 감사위원회와 지배기구의 커뮤니케이션             targets_in_top5=0  


In [8]:
# 상세 top-5 (첫 3개 query 만 스샷 — 나머지는 JSON 덤프 참조)
for r in seed_results[:3]:
    print(f"\n[{r['category']}] {r['query']}")
    for row in r["top5"]:
        mark = "★" if row["chunk_id"] in TARGET_CHUNKS else " "
        print(f"  {mark} #{row['rank']} score={row['score']}  {row['chunk_id']}")
        print(f"       {row['content_preview']}...")


[A] 용어의 정의
    #1 score=0.337  ISA-620:definitions:a60dac4d:6.
       감사기준에서 사용하는 용어의 정의는 다음과 같다....
    #2 score=0.337  ISA-600:definitions:5310730d:9.
       감사기준에서 사용하는 용어의 정의는 다음과 같다....
    #3 score=0.337  ISA-560:definitions:0ebf5093:5.
       감사기준에서 사용하는 용어의 정의는 다음과 같다....
    #4 score=0.337  ISA-580:definitions:d961355a:7.
       감사기준에서 사용하는 용어의 정의는 다음과 같다....
    #5 score=0.337  ISA-706:definitions:92b474da:7.
       감사기준에서 사용하는 용어의 정의는 다음과 같다....

[A] 감사기준서 용어 정의 전체 목록
    #1 score=0.5216  ISA-315:definitions:56d3af17:12.
       이 감사기준서에서 사용하는 용어의 정의는 다음과 같다....
    #2 score=0.5108  ISA-600:definitions:5310730d:9.
       감사기준에서 사용하는 용어의 정의는 다음과 같다....
    #3 score=0.5108  ISA-620:definitions:a60dac4d:6.
       감사기준에서 사용하는 용어의 정의는 다음과 같다....
    #4 score=0.5108  ISA-560:definitions:0ebf5093:5.
       감사기준에서 사용하는 용어의 정의는 다음과 같다....
    #5 score=0.5108  ISA-580:definitions:d961355a:7.
       감사기준에서 사용하는 용어의 정의는 다음과 같다....

[A] 용어 정의 사전
  ★ #1 score=0.3185  ISA-1200:appendix:d3e

## 6. Header bias 판정 (§9.5)

### 조건 A — co-occurrence ratio
10 query 중 top-5 에 TARGET_CHUNKS 2+ 건이 동시 등장하는 비율 **≥30%** 면 TRIGGER.

### 조건 B — 3-part cosine distance
`#11079 / #11079#1 / #11079#2` passage vector 3 조각 상호 cosine 평균 Δ **<0.01** 이면 TRIGGER.

In [9]:
# 조건 A — co-occurrence
co_count = sum(1 for r in seed_results if r["co_occur_2plus"])
co_ratio = co_count / len(seed_results)
cond_a_trigger = co_ratio >= 0.30
print(f"조건 A  co-occurrence ratio: {co_count}/{len(seed_results)} = {co_ratio:.0%}")
print(f"        threshold ≥30%  → {'TRIGGER' if cond_a_trigger else 'PASS'}")

조건 A  co-occurrence ratio: 0/10 = 0%
        threshold ≥30%  → PASS


In [10]:
# 조건 B — 3-part cosine distance
target_ids = list(TARGET_CHUNKS)
point_ids = [chunk_id_to_point_id(cid) for cid in target_ids]
records = client.retrieve(
    collection_name=COLLECTION,
    ids=point_ids,
    with_vectors=True,
    with_payload=True,
)
assert len(records) == 3, f"expected 3 target points, got {len(records)}"
vec_by_id = {rec.payload["chunk_id"]: rec.vector[VECTOR_PASSAGE] for rec in records}

def cosine(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    na = sum(x * x for x in a) ** 0.5
    nb = sum(y * y for y in b) ** 0.5
    return dot / (na * nb) if na and nb else 0.0

pairwise: list[tuple[str, str, float, float]] = []  # (a, b, cos_sim, cos_dist)
for a, b in combinations(target_ids, 2):
    sim = cosine(vec_by_id[a], vec_by_id[b])
    pairwise.append((a, b, sim, 1.0 - sim))

avg_dist = sum(p[3] for p in pairwise) / len(pairwise)
cond_b_trigger = avg_dist < 0.01

print("pairwise cosine:")
for a, b, sim, dist in pairwise:
    a_short = a.rsplit(":", 1)[-1]
    b_short = b.rsplit(":", 1)[-1]
    print(f"  {a_short:25s} ↔ {b_short:25s}  sim={sim:.4f}  Δ={dist:.4f}")
print(f"\n조건 B  평균 cosine Δ: {avg_dist:.4f}")
print(f"        threshold <0.01  → {'TRIGGER' if cond_b_trigger else 'PASS'}")

pairwise cosine:
  table#11079#1             ↔ table#11079                sim=0.8323  Δ=0.1677
  table#11079#1             ↔ table#11079#2              sim=0.7664  Δ=0.2336
  table#11079               ↔ table#11079#2              sim=0.7873  Δ=0.2127

조건 B  평균 cosine Δ: 0.2046
        threshold <0.01  → PASS


In [11]:
# 종합 판정
if cond_a_trigger and cond_b_trigger:
    verdict = "STRONG TRIGGER — v1.2 MINOR bump + Phase 4 RAG deploy 이전 반드시 해소"
elif cond_a_trigger or cond_b_trigger:
    verdict = "TRIGGER — v1.2 MINOR bump (header-suppression rule)"
else:
    verdict = "PASS — §9.5 finding 종결, v1.2 bump 불필요"

print("=== §9.5 header bias 판정 ===")
print(f"조건 A (co-occur ≥30%):  {'✓' if cond_a_trigger else '✗'}")
print(f"조건 B (cosine Δ <0.01): {'✓' if cond_b_trigger else '✗'}")
print(f"\n→ {verdict}")

=== §9.5 header bias 판정 ===
조건 A (co-occur ≥30%):  ✗
조건 B (cosine Δ <0.01): ✗

→ PASS — §9.5 finding 종결, v1.2 bump 불필요


## 7. 결과 저장 — `output/search_demo_results.json`

Task #7 (CP3 검수) 에서 domain reviewer 가 이 결과를 읽어 §9.5 verdict 를 `docs/checkpoint_3_review.md` 에 기록합니다.

In [12]:
output_path = PROJECT_ROOT / "output" / "search_demo_results.json"
output_path.parent.mkdir(parents=True, exist_ok=True)

payload = {
    "collection": COLLECTION,
    "points_count": info.points_count,
    "embedder_stats_after_run": embedder.stats.to_dict(),
    "seed_queries": seed_results,
    "target_chunks": sorted(TARGET_CHUNKS),
    "header_bias": {
        "cond_a_co_occur_count": co_count,
        "cond_a_co_occur_ratio": co_ratio,
        "cond_a_trigger": cond_a_trigger,
        "cond_b_pairwise_distance": [
            {"a": a, "b": b, "cosine_similarity": sim, "cosine_distance": dist}
            for a, b, sim, dist in pairwise
        ],
        "cond_b_avg_distance": avg_dist,
        "cond_b_trigger": cond_b_trigger,
        "verdict": verdict,
    },
}
output_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"결과 저장: {output_path}")
print(f"파일 크기: {output_path.stat().st_size:,} bytes")

결과 저장: /home/shin/Project/_AuditStandard_parsing/output/search_demo_results.json
파일 크기: 31,810 bytes


In [13]:
# Embedder 정리 (SQLite connection close)
embedder.close()
print("done.")

done.


---

## 사용 가이드 (추가 실험 시)

### 개별 query 빠르게 시험
```python
for row in search_top_k("내가 궁금한 질문", k=10):
    print(row["chunk_id"], row["score"], row["content_preview"])
```

### 특정 standard 내에서만 검색
```python
from qdrant_client.models import Filter, FieldCondition, MatchValue
only_isa_315 = Filter(must=[FieldCondition(key="standard_id", match=MatchValue(value="ISA-315"))])
for row in search_top_k("위험평가절차", k=5, filter_=only_isa_315):
    print(row)
```

### Appendix (보론) 만 검색
```python
appendix_only = Filter(must=[FieldCondition(key="section", match=MatchValue(value="appendix"))])
for row in search_top_k("사례 예시", k=5, filter_=appendix_only):
    print(row)
```

### 2-단계 검색 파라미터 튜닝
- `n_standards` (Stage1 후보 수): 기본 3 — 너무 크면 stage2 가 느려지고, 너무 작으면 관련 standard 누락.
- `k_per_standard` (Stage2 per-standard top-k): 기본 5 — RAG 에 넘길 최종 context 개수.

### 캐시 관리
- 질문 임베딩은 `.embed_cache.sqlite` 에 누적 저장. 동일 질문 재실행 시 API 호출 0.
- 캐시 초기화: `rm .embed_cache.sqlite` (주의: 8,590 chunk passage 도 함께 삭제됨).

### 문제 해결

| 증상 | 원인 | 해결 |
|---|---|---|
| `ConnectionError: localhost:6333` | Qdrant 미기동 | `docker compose up -d` |
| `UPSTAGE_API_KEY missing` | `.env` 미로드 | `.env` 파일 존재 확인 + `load_dotenv()` 재실행 |
| `points_count != 8626` | 적재 미완료 | `audit-parser ingest output/md/ --out output/json/ --upsert` 재실행 |
| query 결과 전부 무관한 chunk | embedding_model mismatch | `.embed_cache.sqlite` 삭제 후 전체 재적재 |
